In [19]:
# Consensus-based header repair + drop still-buggy files before computing canon/intersection
# Steps:
# 1) Parse & normalize headers.
# 2) Build a reference header per group (majority vote by index).
# 3) Correct each file by trimming tokens that startwith REF[i]; count suspicious.
# 4) Drop files that still look corrupted (bad chars OR high suspicious ratio).
# 5) On the remaining files: compute canonical order, intersection, and per-file outside_intersection.

import json, re
from collections import Counter
from pathlib import Path

ROOT   = Path(".")  # run from raw/txt
GROUPS = ["hogar", "individual"]
GLOB   = "*.txt"

# ---- parsing helpers ----

def read_header_text(p: Path) -> str:
    with p.open("rb") as fh:
        raw = fh.readline()
    return raw.decode("utf-8", errors="replace").rstrip("\r\n")

def sniff_delimiter_from_text(s: str, candidates=(";", ",", "\t", "|")) -> str:
    counts = {d: s.count(d) for d in candidates}
    best = max(counts.items(), key=lambda kv: kv[1])[0]
    return best if counts[best] > 0 else ";"

_SPACE_RE = re.compile(r"\s+")
def normalize_token(s: str) -> str:
    s = _SPACE_RE.sub(" ", s.strip()).upper()
    # strip wrapping quotes if present
    for _ in range(2):
        if len(s) >= 2 and ((s[0] == s[-1] == '"') or (s[0] == s[-1] == "'")):
            s = s[1:-1].strip()
    s = s.strip('\'"')
    return s

def split_header_simple(s: str, delim: str):
    parts = [normalize_token(x) for x in s.split(delim)]
    parts = [p for p in parts if p]
    # de-dup while preserving order (guards against accidental dup names)
    seen, uniq = set(), []
    for p in parts:
        if p not in seen:
            uniq.append(p); seen.add(p)
    return uniq

def natural_key(p: Path):
    parts = re.split(r"(\d+)", p.name.lower())
    return [int(s) if s.isdigit() else s for s in parts]

def group_of(rel: str):
    return rel.split("/", 1)[0].lower()

# ---- 1) Parse headers ----
columns_by_file = {}  # rel_path -> [tokens]
for grp in GROUPS:
    base = ROOT / grp
    if not base.exists():
        print(f"[warn] missing group dir: {base}")
        continue
    for p in sorted(base.rglob(GLOB), key=natural_key):
        header = read_header_text(p)
        if not header:
            print(f"[warn] empty header: {p}")
            continue
        delim = sniff_delimiter_from_text(header)
        cols  = split_header_simple(header, delim)
        columns_by_file[str(p.relative_to(ROOT))] = cols

Path("columns_by_file.json").write_text(json.dumps(columns_by_file, indent=2, ensure_ascii=False))
print(f"Indexed {len(columns_by_file)} files → columns_by_file.json")

# ---- 2) Build reference header (majority per index) ----
def build_reference_for_group(files):
    length_counts = Counter(len(columns_by_file[f]) for f in files)
    target_len, _ = max(length_counts.items(), key=lambda kv: kv[1])
    aligned = [f for f in files if len(columns_by_file[f]) == target_len]
    if not aligned:
        target_len = max(len(columns_by_file[f]) for f in files)
        aligned = [f for f in files if len(columns_by_file[f]) == target_len]
    ref = []
    for i in range(target_len):
        votes = Counter(columns_by_file[f][i] for f in aligned)
        max_count = max(votes.values())
        candidates = [tok for tok, c in votes.items() if c == max_count]
        pick = min(candidates, key=len)  # tie-breaker: shortest token wins
        ref.append(pick)
    return ref, aligned

reference_headers = {}
for grp in GROUPS:
    files = [f for f in sorted(columns_by_file, key=lambda s: s.lower()) if group_of(f)==grp]
    if not files:
        continue
    ref, used = build_reference_for_group(files)
    reference_headers[grp] = ref
    Path(f"reference_header_{grp}.json").write_text(json.dumps(ref, indent=2, ensure_ascii=False))
    print(f"[{grp.upper()}] reference length={len(ref)} (from {len(used)} aligned files) → reference_header_{grp}.json")

# ---- 3) Correct tokens by position using the reference + measure quality ----
corrected_by_file = {}
quality = {}  # rel_path -> dict(stats)
BAD_TOKEN_RE = re.compile(r"[^A-Z0-9_]")  # any char outside allowed set

for grp in GROUPS:
    if grp not in reference_headers:
        continue
    ref = reference_headers[grp]
    files = [f for f in sorted(columns_by_file, key=lambda s: s.lower()) if group_of(f)==grp]

    for f in files:
        raw = columns_by_file[f]
        corrected = []
        trimmed = 0
        suspicious = 0
        comparable = min(len(raw), len(ref))

        for i in range(comparable):
            tok = raw[i]
            want = ref[i]
            if tok == want:
                corrected.append(tok)
            elif tok.startswith(want):
                corrected.append(want)
                trimmed += 1
            elif want.startswith(tok):
                corrected.append(want)
                trimmed += 1
            else:
                corrected.append(tok)
                suspicious += 1

        # keep any extra trailing tokens (count them as suspicious)
        if len(raw) > len(ref):
            corrected.extend(raw[len(ref):])
            suspicious += len(raw) - len(ref)

        corrected_by_file[f] = corrected

        # token cleanliness after correction
        dirty_tokens = sum(1 for t in corrected if BAD_TOKEN_RE.search(t))
        suspicious_ratio = (suspicious / max(1, comparable))

        quality[f] = {
            "group": grp,
            "raw_cols": len(raw),
            "corrected_cols": len(corrected),
            "trimmed": trimmed,
            "suspicious": suspicious,
            "suspicious_ratio": round(suspicious_ratio, 4),
            "dirty_tokens": dirty_tokens,
        }

Path("corrected_columns_by_file.json").write_text(json.dumps(corrected_by_file, indent=2, ensure_ascii=False))
Path("header_quality.json").write_text(json.dumps(quality, indent=2, ensure_ascii=False))
print("→ wrote corrected_columns_by_file.json, header_quality.json")

# ---- 4) Drop still-buggy files (dirty tokens OR suspicious ratio > threshold) ----
SUSP_RATIO_THRESH = 1  # 15% unmatched to reference is considered corrupted

kept_files = {g: [] for g in GROUPS}
dropped_files = {g: [] for g in GROUPS}

for f, q in quality.items():
    grp = q["group"]
    drop = (q["dirty_tokens"] > 0) or (q["suspicious_ratio"] > SUSP_RATIO_THRESH)
    (dropped_files if drop else kept_files)[grp].append({"file": f, **q})

Path("dropped_files.json").write_text(json.dumps(dropped_files, indent=2, ensure_ascii=False))
Path("kept_files.json").write_text(json.dumps(kept_files, indent=2, ensure_ascii=False))

print("\nDrop summary (by group):")
for grp in GROUPS:
    k = len(kept_files.get(grp, []))
    d = len(dropped_files.get(grp, []))
    print(f"- {grp}: kept={k}, dropped={d} (threshold={SUSP_RATIO_THRESH*100:.0f}% suspicious or any dirty tokens)")

# ---- 5) On kept files only: build canon, intersection, and per-file outside_intersection ----
artifacts = []
for grp in GROUPS:
    files_all = [f for f in corrected_by_file if group_of(f)==grp]
    files = [item["file"] for item in kept_files.get(grp, [])]
    if not files:
        print(f"[{grp}] no kept files (all dropped?)")
        continue

    # Rebuild canon from kept files only
    def build_reference_from_corrected(files):
        lengths = Counter(len(corrected_by_file[f]) for f in files)
        target_len, _ = max(lengths.items(), key=lambda kv: kv[1])
        aligned = [f for f in files if len(corrected_by_file[f]) == target_len] \
                  or [max(files, key=lambda f: len(corrected_by_file[f]))]
        ref = []
        for i in range(target_len):
            votes = Counter(corrected_by_file[f][i] for f in aligned if i < len(corrected_by_file[f]))
            max_count = max(votes.values())
            cand = [tok for tok, c in votes.items() if c == max_count]
            ref.append(min(cand, key=len))
        return ref

    canon = build_reference_from_corrected(files)

    # Intersection over kept files, ordered by canon
    sets = [set(corrected_by_file[f]) for f in files]
    inter = [c for c in canon if all(c in s for s in sets)]

    # Per-file outside_intersection
    lines = [f"# {grp.upper()} — intersection (kept files) + per-file diffs\n"]
    lines.append(f"## Intersection ({len(inter)})\n")
    lines.append(", ".join(inter) + "\n")

    for f in sorted(files, key=lambda s: re.split(r'(\d+)', s.lower())):
        cols = corrected_by_file[f]
        outside = [c for c in cols if c not in inter]
        q = next((item for item in kept_files[grp] if item["file"] == f), None) or {}
        lines.append(f"### {f}")
        lines.append(f"- total_columns: {len(cols)}")
        lines.append(f"- outside_intersection ({len(outside)}):")
        lines.append("  - " + (", ".join(outside) if outside else "(none)"))
        lines.append(f"- quality: suspicious={q.get('suspicious')} "
                     f"({q.get('suspicious_ratio')}), dirty_tokens={q.get('dirty_tokens')}")
        lines.append("")

    # write artifacts
    inter_path = Path(f"intersection_{grp}_kept.txt")
    diffs_path = Path(f"intersection_and_diffs_{grp}_kept.md")
    inter_path.write_text("\n".join(inter), encoding="utf-8")
    diffs_path.write_text("\n".join(lines), encoding="utf-8")
    artifacts += [str(inter_path), str(diffs_path)]

print("\nwrote:", ", ".join(artifacts))
print("See also: dropped_files.json and kept_files.json for reasons/files.")


Indexed 163 files → columns_by_file.json
[HOGAR] reference length=88 (from 80 aligned files) → reference_header_hogar.json
[INDIVIDUAL] reference length=177 (from 34 aligned files) → reference_header_individual.json
→ wrote corrected_columns_by_file.json, header_quality.json

Drop summary (by group):
- hogar: kept=74, dropped=8 (threshold=100% suspicious or any dirty tokens)
- individual: kept=74, dropped=7 (threshold=100% suspicious or any dirty tokens)

wrote: intersection_hogar_kept.txt, intersection_and_diffs_hogar_kept.md, intersection_individual_kept.txt, intersection_and_diffs_individual_kept.md
See also: dropped_files.json and kept_files.json for reasons/files.


In [20]:
# Use corrected headers → build canonical order, intersection, and per-file diffs
import json, re
from collections import Counter
from pathlib import Path

ROOT = Path(".")
IN = Path("corrected_columns_by_file.json")
assert IN.exists(), f"missing {IN}"

with IN.open("r", encoding="utf-8") as fh:
    corrected = json.load(fh)

def group_of(rel: str) -> str:
    return rel.split("/", 1)[0].lower()

def natural_key(name: str):
    parts = re.split(r"(\d+)", name.lower())
    return [int(s) if s.isdigit() else s for s in parts]

def build_reference(files):
    lengths = Counter(len(corrected[f]) for f in files)
    target_len, _ = max(lengths.items(), key=lambda kv: kv[1])
    aligned = [f for f in files if len(corrected[f]) == target_len] or \
              [max(files, key=lambda f: len(corrected[f]))]  # fallback: longest only
    ref = []
    for i in range(target_len):
        votes = Counter(corrected[f][i] for f in aligned if i < len(corrected[f]))
        max_count = max(votes.values())
        cand = [tok for tok, c in votes.items() if c == max_count]
        ref.append(min(cand, key=len))  # tie-break: shortest wins
    return ref

artifacts = []

for grp in ["hogar", "individual"]:
    files = sorted([f for f in corrected if group_of(f) == grp], key=natural_key)
    if not files:
        print(f"[{grp}] no files found")
        continue

    # 1) Canonical order by index majority vote
    canon = build_reference(files)

    # 2) Intersection across all files, ordered by canon
    sets = [set(corrected[f]) for f in files]
    inter = list(dict.fromkeys([c for c in canon if all(c in s for s in sets)]))

    # 3) Per-file diff outside intersection (preserve file order)
    out_lines = [f"# {grp.upper()} — intersection + per-file diffs\n"]
    out_lines.append(f"## Intersection ({len(inter)})\n")
    out_lines.append(", ".join(inter) + "\n")

    for f in files:
        cols = corrected[f]
        outside = [c for c in cols if c not in inter]
        out_lines.append(f"### {f}")
        out_lines.append(f"- total_columns: {len(cols)}")
        out_lines.append(f"- outside_intersection ({len(outside)}):")
        out_lines.append("  - " + (", ".join(outside) if outside else "(none)"))
        out_lines.append("")

        # concise console line
        print(f"{grp}: {f}: outside_intersection={len(outside)} (total={len(cols)})")

    # Write artifacts
    inter_path = Path(f"intersection_{grp}.txt")
    diffs_path = Path(f"intersection_and_diffs_{grp}.md")
    inter_path.write_text("\n".join(inter), encoding="utf-8")
    diffs_path.write_text("\n".join(out_lines), encoding="utf-8")
    artifacts += [str(inter_path), str(diffs_path)]

print("wrote:", ", ".join(artifacts))


hogar: hogar/EPH_usu_hogar_4to_trim2020_txt.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t104.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t105.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t106.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t107.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t109.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t204.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t205.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t206.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t207.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t208.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t209.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t303.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t304.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t305.txt: outside_intersection=6 (total=88)
hogar: hogar/Hog_t306.txt: outside_intersection=6 (total=88)
ho

In [ ]:
s = read_header_text(Path("individual/Individual_t314.txt"))
print(" ".join(f"U+{ord(ch):04X}:{repr(ch)}" for ch in s[:300]))

# U+0043:'C' U+004F:'O' U+0044:'D' U+0055:'U' U+0053:'S' U+0055:'U' U+004E:'N' U+003B:';' U+004E:'N' U+0052:'R' U+004F:'O' U+005F:'_' U+0048:'H' U+004F:'O' U+0047:'G' U+0041:'A' U+0052:'R' U+003B:';' U+0043:'C' U+004F:'O' U+004D:'M' U+0050:'P' U+004F:'O' U+004E:'N' U+0045:'E' U+004E:'N' U+0054:'T' U+0045:'E' U+003B:';' U+0048:'H' U+0031:'1' U+0035:'5' U+004F:'O' U+004E:'N' U+0045:'E' U+004E:'N' U+0054:'T' U+0045:'E' U+003B:';' U+0041:'A' U+004E:'N' U+004F:'O' U+0034:'4' U+0041:'A' U+004E:'N' U+0041:'A' U+003B:';' U+0054:'T' U+0052:'R' U+0049:'I' U+004D:'M' U+0045:'E' U+0053:'S' U+0054:'T' U+0052:'R' U+0045:'E' U+003B:';' U+0052:'R' U+0045:'E' U+0047:'G' U+0049:'I' U+004F:'O' U+004E:'N' U+004E:'N' U+003B:';' U+004D:'M' U+0041:'A' U+0053:'S' U+005F:'_' U+0035:'5' U+0030:'0' U+0030:'0' U+003B:';' U+0041:'A' U+0047:'G' U+004C:'L' U+004F:'O' U+004D:'M' U+0045:'E' U+0052:'R' U+0041:'A' U+0044:'D' U+004F:'O' U+003B:';' U+0050:'P' U+004F:'O' U+004E:'N' U+0044:'D' U+0045:'E' U+0052:'R' U+0041:'A' U+0045:'E' U+003B:';' U+0043:'C' U+0048:'H' U+0030:'0' U+0033:'3' U+004E:'N' U+0044:'D' U+003B:';' U+0043:'C' U+0048:'H' U+0030:'0' U+0034:'4' U+0030:'0' U+0030:'0' U+003B:';' U+0043:'C' U+0048:'H' U+0030:'0' U+0036:'6' U+0052:'R' U+0041:'A' U+003B:';' U+0043:'C' U+0048:'H' U+0030:'0' U+0037:'7' U+0043:'C' U+004F:'O' U+0044:'D' U+003B:';' U+0043:'C' U+0048:'H' U+0030:'0' U+0038:'8' U+0043:'C' U+004F:'O' U+0044:'D' U+003B:';' U+0043:'C' U+0048:'H' U+0030:'0' U+0039:'9' U+0043:'C' U+004F:'O' U+0044:'D' U+003B:';' U+0043:'C' U+0048:'H' U+0031:'1' U+0030:'0' U+0043:'C' U+004F:'O' U+0044:'D' U+003B:';' U+0043:'C' U+0048:'H' U+0031:'1' U+0031:'1' U+0043:'C' U+004F:'O' U+0044:'D' U+003B:';' U+0043:'C' U+0048:'H' U+0031:'1' U+0032:'2' U+0043:'C' U+004F:'O' U+0044:'D' U+003B:';' U+0043:'C' U+0048:'H' U+0031:'1' U+0033:'3' U+005F:'_' U+0045:'E' U+0044:'D' U+003B:';' U+0043:'C' U+0048:'H' U+0031:'1' U+0034:'4' U+004F:'O' U+0050:'P' U+003B:';' U+0043:'C' U+0048:'H' U+0031:'1' U+0035:'5' U+0043:'C' U+0055:'U' U+0050:'P' U+0053:'S' U+003B:';' U+0043:'C' U+0048:'H' U+0031:'1' U+0035:'5' U+005F:'_' U+0043:'C' U+004F:'O' U+0044:'D' U+0053:'S' U+003B:';' U+0043:'C' U+0048:'H' U+0031:'1' U+0036:'6' U+0031:'1' U+0054:'T' U+0053:'S' U+003B:';' U+0043:'C' U+0048:'H' U+0031:'1' U+0036:'6' U+005F:'_' U+0043:'C' U+004F:'O' U+0044:'D' U+0053:'S' U+003B:';' U+004E:'N' U+0049:'I' U+0056:'V' U+0045:'E' U+004C:'L' U+005F:'_' U+0045:'E' U+0044:'D' U+0053:'S' U+003B:';' U+0045:'E' U+0053:'S' U+0054:'T' U+0041:'A' U+0044:'D' U+004F:'O' U+0044:'D' U+0053:'S' U+003B:';' U+0043:'C' U+0041:'A' U+0054:'T' U+005F:'_' U+004F:'O' U+0043:'C' U+0055:'U' U+0050:'P' U+0053:'S' U+003B:';' U+0043:'C' U+0041:'A' U+0054:'T' U+005F:'_' U+0049:'I' U+004E:'N' U+0041:'A' U+0043:'C' U+0053:'S' U+003B:';' U+0050:'P' U+0050:'P' U+0030:'0' U+0032:'2' U+0043:'C' U+0031:'1' U+0043:'C' U+0053:'S' U+003B:';' U+0050:'P' U+0050:'P' U+0030:'0' U+0032:'2' U+0043:'C' U+0032:'2' U+0054:'T' U+0053:'S' U+003B:';' U+0050:'P' U+0050:'P' U+0030:'0' U+0032:'2' U+0043:'C' U+0033:'3' U+0054:'T' U+0053:'S' U+003B:';' U+0050:'P' U+0050:'P' U+0030:'0' U+0032:'2' U+0043:'C' U+0034:'4' U+0054:'T' U+0053:'S' U+003B:';' U+0050:'P' U+0050:'P' U+0030:'0' U+0032:'2' U+0043:'C' U+0035:'5' U+0054:'T' U+0053:'S' U+003B:';' U+0050:'P' U+0050:'P'

In [27]:
# Drop-on-sight for buggy headers (incl. explicit blacklist), then canon + intersection + per-file diffs

import json, re
from collections import Counter
from pathlib import Path

ROOT   = Path(".")  # run from raw/txt
GROUPS = ["hogar", "individual"]
GLOB   = "*.txt"

# ---------- parsing ----------
def read_header_text(p: Path) -> str:
    with p.open("rb") as fh:
        raw = fh.readline()
    return raw.decode("utf-8", errors="replace").rstrip("\r\n")

def sniff_delimiter_from_text(s: str, candidates=(";", ",", "\t", "|")) -> str:
    counts = {d: s.count(d) for d in candidates}
    best = max(counts.items(), key=lambda kv: kv[1])[0]
    return best if counts[best] > 0 else ";"

_SPACE_RE = re.compile(r"\s+")
def normalize_token(s: str) -> str:
    s = _SPACE_RE.sub(" ", s.strip()).upper()
    for _ in range(2):
        if len(s) >= 2 and ((s[0] == s[-1] == '"') or (s[0] == s[-1] == "'")):
            s = s[1:-1].strip()
    s = s.strip('\'"')
    return s

def split_header_simple(s: str, delim: str):
    parts = [normalize_token(x) for x in s.split(delim)]
    parts = [p for p in parts if p]
    seen, uniq = set(), []
    for p in parts:
        if p not in seen:
            uniq.append(p); seen.add(p)
    return uniq

def natural_key(p: Path):
    parts = re.split(r"(\d+)", p.name.lower())
    return [int(s) if s.isdigit() else s for s in parts]

def group_of(rel: str):
    return rel.split("/", 1)[0].lower()

BAD_TOKEN_RE = re.compile(r"[^A-Z0-9_]")  # anything outside allowed set

# Explicit blacklist
BLACKLIST = {
    "individual/Individual_t414.txt",
    "individual/Individual_t314.txt",
    "individual/Individual_t215.txt",
    "individual/individual_t214.txt",
    "individual/Individual_t115.txt",
    "individual/Individual_t114.txt",
    "individual/Individual_t413.txt",
    "hogar/Hogar_t114.txt",
    "hogar/Hogar_t115.txt",
    "hogar/Hogar_t414.txt",
    "hogar/hogar_t214.txt",
    "hogar/Hogar_t215.txt",
}

# ---------- 1) Parse headers + DROP buggy or blacklisted files ----------
columns_by_file_all = {}
columns_by_file     = {}
dropped             = {g: [] for g in GROUPS}

for grp in GROUPS:
    base = ROOT / grp
    if not base.exists():
        print(f"[warn] missing group dir: {base}")
        continue

    for p in sorted(base.rglob(GLOB), key=natural_key):
        rel = str(p.relative_to(ROOT))

        if rel in BLACKLIST:
            dropped[grp].append({"file": rel, "reason": "blacklisted"})
            continue

        header = read_header_text(p)
        if not header:
            dropped[grp].append({"file": rel, "reason": "empty header"})
            continue

        delim = sniff_delimiter_from_text(header)
        cols  = split_header_simple(header, delim)
        columns_by_file_all[rel] = cols

        if any(BAD_TOKEN_RE.search(tok) for tok in cols):
            dropped[grp].append({"file": rel, "reason": "bad characters in tokens"})
            continue

        columns_by_file[rel] = cols

Path("columns_by_file.ALL.json").write_text(json.dumps(columns_by_file_all, indent=2, ensure_ascii=False))
Path("columns_by_file.json").write_text(json.dumps(columns_by_file, indent=2, ensure_ascii=False))
Path("dropped_files_early.json").write_text(json.dumps(dropped, indent=2, ensure_ascii=False))
print(f"Parsed {len(columns_by_file_all)} files → kept {len(columns_by_file)} (dropped {sum(len(v) for v in dropped.values())})")

# ---------- 2) Build reference (majority-by-index) on kept files ----------
def build_reference_for_group(files):
    length_counts = Counter(len(columns_by_file[f]) for f in files)
    if not length_counts:
        return [], []
    target_len, _ = max(length_counts.items(), key=lambda kv: kv[1])
    aligned = [f for f in files if len(columns_by_file[f]) == target_len]
    if not aligned:
        target_len = max(len(columns_by_file[f]) for f in files)
        aligned = [f for f in files if len(columns_by_file[f]) == target_len]
    ref = []
    for i in range(target_len):
        votes = Counter(columns_by_file[f][i] for f in aligned if i < len(columns_by_file[f]))
        max_count = max(votes.values())
        candidates = [tok for tok, c in votes.items() if c == max_count]
        ref.append(min(candidates, key=len))
    return ref, aligned

reference_headers = {}
for grp in GROUPS:
    files = sorted([f for f in columns_by_file if group_of(f) == grp], key=lambda s: re.split(r"(\d+)", s.lower()))
    if not files:
        print(f"[{grp}] no clean files kept (all dropped?)")
        continue
    ref, used = build_reference_for_group(files)
    reference_headers[grp] = ref
    Path(f"reference_header_{grp}.json").write_text(json.dumps(ref, indent=2, ensure_ascii=False))
    print(f"[{grp.upper()}] reference length={len(ref)} (from {len(used)} aligned clean files)")

# ---------- 3) Trivial correction ----------
corrected_by_file = {}
for grp in GROUPS:
    ref = reference_headers.get(grp, [])
    files = [f for f in columns_by_file if group_of(f) == grp]
    for f in files:
        raw = columns_by_file[f]
        if not ref:
            corrected_by_file[f] = raw
            continue
        n = min(len(raw), len(ref))
        corr = []
        for i in range(n):
            tok, want = raw[i], ref[i]
            corr.append(want if tok.startswith(want) or want.startswith(tok) else tok)
        if len(raw) > n:
            corr.extend(raw[n:])
        corrected_by_file[f] = corr

Path("corrected_columns_by_file.json").write_text(json.dumps(corrected_by_file, indent=2, ensure_ascii=False))

# ---------- 4) Intersection + per-file outside_intersection ----------
artifacts = []
for grp in GROUPS:
    files = sorted([f for f in corrected_by_file if group_of(f) == grp], key=lambda s: re.split(r"(\d+)", s.lower()))
    if not files:
        continue
    canon = reference_headers.get(grp, [])
    sets = [set(corrected_by_file[f]) for f in files]
    inter = [c for c in canon if all(c in s for s in sets)]
    inter_path = Path(f"intersection_{grp}_kept.txt")
    diffs_path = Path(f"intersection_and_diffs_{grp}_kept.md")
    lines = [
        f"# {grp.upper()} — intersection (clean files only) + per-file diffs\n",
        f"## Intersection ({len(inter)})\n",
        ", ".join(inter) + "\n"
    ]
    for f in files:
        cols = corrected_by_file[f]
        outside = [c for c in cols if c not in inter]
        lines += [
            f"### {f}",
            f"- total_columns: {len(cols)}",
            f"- outside_intersection ({len(outside)}):",
            "  - " + (", ".join(outside) if outside else "(none)"),
            ""
        ]
    inter_path.write_text("\n".join(inter), encoding="utf-8")
    diffs_path.write_text("\n".join(lines), encoding="utf-8")
    artifacts += [str(inter_path), str(diffs_path)]

print("wrote:", ", ".join(artifacts))
print("See dropped files with reasons in: dropped_files_early.json")


Parsed 151 files → kept 135 (dropped 28)
[HOGAR] reference length=88 (from 66 aligned clean files)
[INDIVIDUAL] reference length=177 (from 34 aligned clean files)
wrote: intersection_hogar_kept.txt, intersection_and_diffs_hogar_kept.md, intersection_individual_kept.txt, intersection_and_diffs_individual_kept.md
See dropped files with reasons in: dropped_files_early.json
